# DART Data Assimilation with CESM-MOM6

This tutorial walks through setting up and running ensemble data assimilation using
[DART](https://dart.ucar.edu) (Data Assimilation Research Testbed) with a regional MOM6
ocean model configured through the CROCODILE CESM framework.

A typical DART–CESM workflow consists of four main steps:

1. Configure the CESM-MOM6 case for assimilation (ensemble size, DART namelists).
2. Prepare observations in DART obs_sequence format.
3. Run the assimilation cycle (forecast → assimilate → update state).
4. Examine and diagnose assimilation output.

**Prerequisites**: A working CESM-MOM6 regional configuration. Follow the
[CrocoDash Getting Started](../tutorials/crocodash_tutorial.ipynb) tutorial first if needed.

# SECTION 1: Configure the CESM Case for DART

DART is tightly coupled to CESM. The assimilation is driven by the `CESM_interface` scripts
that live inside the DART source tree. This section covers:
- Pointing to your existing CESM-MOM6 case
- Setting the ensemble size
- Configuring the DART `input.nml` namelist

## Step 1.1: Set paths

In [ ]:
from pathlib import Path

# Path to your DART installation
dart_root = Path("<DART_ROOT>")

# Path to your CESM case directory
cesm_case = Path("<CESM_CASE>")

# Ensemble size
N_ENS = 80

print(f"DART root : {dart_root}")
print(f"CESM case : {cesm_case}")
print(f"Ensemble  : {N_ENS} members")

## Step 1.2: Configure DART namelists

The main DART configuration lives in `input.nml`. Key sections for MOM6:
- `&filter_nml` — ensemble filter settings (EAKF, inflation, localisation)
- `&model_nml` — MOM6-specific state vector variables
- `&obs_kind_nml` — which observation types to assimilate

In [ ]:
# TODO: write / inspect input.nml for the MOM6 DART interface
input_nml = dart_root / "models" / "MOM6" / "work" / "input.nml"
print(input_nml.read_text())

# SECTION 2: Prepare Observations

DART ingests observations through `obs_sequence` files. This section converts
in-situ or satellite observations into the DART native format.

## Step 2.1: Choose an observation source

Common ocean observation types for regional MOM6 DA:

| Observation | DART obs type | Source |
|---|---|---|
| SST | `SATELLITE_INFRARED_SST` | GHRSST / OSTIA |
| SSH | `ALTIMETER` | CMEMS |
| T/S profiles | `ARGO_TEMPERATURE` / `ARGO_SALINITY` | Argo / CrocoLake |
| Sea ice | `SEA_ICE_CONCENTRATION` | NSIDC |

## Step 2.2: Convert observations to obs_sequence format

DART provides observation converters in `DART/observations/obs_converters/`.
Run the appropriate converter for your data source, then verify the output.

In [ ]:
import subprocess

obs_seq_out = Path("<DART_OBS_SEQ>")

# Example: verify a converted obs_sequence file
result = subprocess.run(
    [str(dart_root / "assimilation_code" / "programs" / "obs_diag" / "threed_sphere" / "obs_diag")],
    capture_output=True, text=True
)
print(result.stdout[:2000])

# SECTION 3: Run the Assimilation Cycle

The DART–CESM coupling works through a driver script that:
1. Advances each ensemble member (CESM forecast step)
2. Calls `filter` to assimilate observations and update the ensemble
3. Advances to the next assimilation window

## Step 3.1: Stage the ensemble

Each ensemble member needs its own CESM restart files. The DART `CESM_interface`
scripts handle staging automatically once the case is configured.

In [ ]:
# TODO: show how to stage ensemble restart files
# e.g. using DART/models/MOM6/shell_scripts/stage_cesm_files
pass

## Step 3.2: Submit the assimilation job

On an HPC system (e.g. Derecho), the assimilation cycle is submitted via the
DART `assimilate.csh` script, which is called from the CESM `case.submit` workflow.

In [ ]:
# TODO: example qsubmit / PBS job submission for the DA cycle
pass

# SECTION 4: Examine Assimilation Output

After `filter` runs, DART writes diagnostic NetCDF files:
- `obs_diag_output.nc` — observation-space diagnostics (prior/posterior fits)
- `preassim_mean.nc` / `postassim_mean.nc` — ensemble mean state before/after assimilation
- `output_mean.nc` — updated ensemble mean written back to CESM restart form

## Step 4.1: Observation-space diagnostics

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

obs_diag = xr.open_dataset("<DART_OUTPUT>/obs_diag_output.nc")

# Plot prior and posterior RMSE for SST
# TODO: filter for SST obs kind and plot time series
print(obs_diag)

## Step 4.2: State-space increments

In [ ]:
preassim = xr.open_dataset("<DART_OUTPUT>/preassim_mean.nc")
postassim = xr.open_dataset("<DART_OUTPUT>/postassim_mean.nc")

# Sea surface temperature increment
increment = postassim["Temp"].isel(Time=0, zl=0) - preassim["Temp"].isel(Time=0, zl=0)
increment.plot(cmap="RdBu_r", robust=True)
plt.title("SST increment (posterior − prior)")
plt.show()